# Analyze Task Leave-Out Hyperalignment

For each held-out task, the alignment was fit on all OTHER tasks.  
This notebook loads the resulting aligned arrays and runs:

1. **ISC validation** — pairwise inter-subject correlation on the aligned held-out data  
   (should be higher than unaligned ISC if alignment generalises across tasks)
2. **Practice-effect slopes** — per-voxel linear regression of activation ~ centered encounter  
   (per subject, then averaged across subjects with a one-sample t-test)

In [ ]:
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from scipy.stats import ttest_1samp
from nilearn import plotting, datasets
import nibabel as nib

try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))
from config import (
    TASKS, CONTRASTS, SUBJECTS, ENCOUNTERS, RT_CONTRAST,
    ENCOUNTER_CENTER, RESULTS_DIR, RSA_CONTRASTS,
    setup_masker_and_labels, BASE_DIR, SESSIONS,
    build_first_level_contrast_map_path,
)

LEAVEOUT_DIR = RESULTS_DIR / 'task_leaveout'
FIG_DIR      = _nb_dir / 'figures' / 'task_leaveout'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Setup done.')

In [ ]:
# Set up masker (needed for inverse_transform → NIfTI brain maps)
reference_path = None
for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        for session in SESSIONS:
            p = build_first_level_contrast_map_path(BASE_DIR, SUBJECTS[0], session, task, contrast)
            if Path(p).exists():
                reference_path = p
                break
        if reference_path: break
    if reference_path: break

masker, labels = setup_masker_and_labels(nib.load(reference_path))
print(f'Masker ready.  Reference: {reference_path}')

In [ ]:
# Load all task leave-out results into memory
# Structure: data[task] = {'masked': {subj: arr}, 'aligned': {subj: arr}, 'tce_order': list}
data = {}
for task in TASKS:
    task_dir = LEAVEOUT_DIR / task
    if not task_dir.exists():
        print(f'  Missing: {task}')
        continue
    with open(task_dir / 'tce_order.pkl', 'rb') as f:
        tce_order = pickle.load(f)
    data[task] = {
        'tce_order': tce_order,
        'masked':  {s: np.load(task_dir / f'masked_{s}.npy')  for s in SUBJECTS},
        'aligned': {s: np.load(task_dir / f'aligned_{s}.npy') for s in SUBJECTS},
    }
    print(f'  {task}: {len(tce_order)} rows, shape {data[task]["aligned"][SUBJECTS[0]].shape}')

## 1. ISC Validation

For each held-out task, compute pairwise Pearson r per voxel across all rows  
(all encounters × contrasts of that task), then average across pairs.

In [ ]:
def voxelwise_pearson(A, B):
    """Per-voxel Pearson r between two (n_rows, n_voxels) arrays."""
    A = A - A.mean(axis=0, keepdims=True)
    B = B - B.mean(axis=0, keepdims=True)
    denom = np.sqrt((A**2).sum(0) * (B**2).sum(0)) + 1e-10
    return (A * B).sum(0) / denom

isc_results = {}   # task -> {'before': mean_r, 'after': mean_r, 'diff_map': (n_voxels,)}
pairs = list(combinations(SUBJECTS, 2))

for task, d in data.items():
    before_vals, after_vals = [], []
    for s1, s2 in pairs:
        r_before = voxelwise_pearson(d['masked'][s1],  d['masked'][s2])
        r_after  = voxelwise_pearson(d['aligned'][s1], d['aligned'][s2])
        before_vals.append(r_before)
        after_vals.append(r_after)
    isc_before = np.mean(before_vals, axis=0)
    isc_after  = np.mean(after_vals,  axis=0)
    isc_results[task] = {
        'before':   isc_before,
        'after':    isc_after,
        'diff_map': isc_after - isc_before,
    }
    print(f'{task:20s}  before={isc_before.mean():.3f}  after={isc_after.mean():.3f}  '
          f'gain={( isc_after - isc_before).mean():.3f}')

In [ ]:
# Bar chart: mean ISC before vs after, per task
tasks_with_data = list(isc_results.keys())
x = np.arange(len(tasks_with_data))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w/2, [isc_results[t]['before'].mean() for t in tasks_with_data], w,
       label='Before', color='steelblue')
ax.bar(x + w/2, [isc_results[t]['after'].mean()  for t in tasks_with_data], w,
       label='After',  color='coral')
ax.set_xticks(x); ax.set_xticklabels(tasks_with_data, rotation=30, ha='right')
ax.set_ylabel('Mean ISC (Pearson r)'); ax.legend()
ax.set_title('ISC before vs after alignment (task leave-out, cross-validated)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'isc_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Brain maps: ISC gain (after - before) per task
for task in tasks_with_data:
    diff_img = masker.inverse_transform(isc_results[task]['diff_map'])
    vmax = np.percentile(np.abs(isc_results[task]['diff_map']), 98)
    display = plotting.plot_stat_map(
        diff_img, display_mode='z', cut_coords=6,
        cmap='RdBu_r', vmax=vmax, colorbar=True,
        title=f'ISC gain: {task}',
    )
    display.savefig(FIG_DIR / f'isc_gain_{task}.png')
    plt.close('all')

## 2. Practice-Effect Slopes

For each held-out task:
- Reshape the aligned array into `(n_contrasts, n_encounters, n_voxels)`
- For each contrast and each subject, fit `activation ~ centered_encounter` per voxel
- Average slopes across subjects with a one-sample t-test (mask p > 0.05)
- Plot group-average slope map on the brain

In [ ]:
from collections import defaultdict

enc_x = np.array([float(e) - ENCOUNTER_CENTER for e in ENCOUNTERS])  # [-2,-1,0,1,2]
X_ols = np.column_stack([np.ones(len(ENCOUNTERS)), enc_x])            # design matrix

slope_results = {}  # task -> contrast -> {'subj_slopes': (n_subj, n_voxels), 'group_slope': (n_voxels,)}

for task, d in data.items():
    tce_order = d['tce_order']  # list of (task, contrast, enc)
    contrasts_in_task = [c for c in CONTRASTS[task] if c != RT_CONTRAST]
    slope_results[task] = {}

    for contrast in contrasts_in_task:
        # Find row indices for this contrast, in encounter order
        enc_indices = {}
        for row_idx, (t, c, e) in enumerate(tce_order):
            if c == contrast:
                enc_indices[e] = row_idx

        # Need all 5 encounters to fit the slope
        if not all(e in enc_indices for e in ENCOUNTERS):
            continue

        ordered_rows = [enc_indices[e] for e in ENCOUNTERS]

        subj_slopes = []
        for subj in SUBJECTS:
            arr = d['aligned'][subj]                       # (n_rows, n_voxels)
            maps = arr[ordered_rows, :]                    # (5, n_voxels)
            B, _, _, _ = np.linalg.lstsq(X_ols, maps, rcond=None)
            subj_slopes.append(B[1])                       # slope row, (n_voxels,)

        subj_slopes = np.array(subj_slopes)                # (n_subj, n_voxels)
        group_slope = subj_slopes.mean(axis=0)
        slope_results[task][contrast] = {
            'subj_slopes':  subj_slopes,
            'group_slope':  group_slope,
        }

    print(f'{task}: fitted slopes for {len(slope_results[task])} contrasts')

In [ ]:
# Per-subject slope maps
indiv_fig_dir = FIG_DIR / 'slopes' / 'individual'
indiv_fig_dir.mkdir(parents=True, exist_ok=True)

for task, contrasts in slope_results.items():
    for contrast, res in contrasts.items():
        for s_idx, subj in enumerate(SUBJECTS):
            slope_img = masker.inverse_transform(res['subj_slopes'][s_idx])
            vmax = np.percentile(np.abs(res['subj_slopes'][s_idx]), 98)
            display = plotting.plot_stat_map(
                slope_img, display_mode='z', cut_coords=6,
                cmap='RdBu_r', vmax=vmax, colorbar=True,
                title=f'{subj} | {task} | {contrast}',
            )
            safe = contrast.replace('/', '-')
            display.savefig(indiv_fig_dir / f'{subj}_{task}_{safe}.png')
            plt.close('all')

print(f'Individual slope maps saved to {indiv_fig_dir}')

In [ ]:
# Group-average slope maps (p < 0.05 mask via one-sample t-test)
group_fig_dir = FIG_DIR / 'slopes' / 'group'
group_fig_dir.mkdir(parents=True, exist_ok=True)

for task, contrasts in slope_results.items():
    for contrast, res in contrasts.items():
        _, p_vals = ttest_1samp(res['subj_slopes'], popmean=0, axis=0)
        masked_slope = res['group_slope'].copy()
        masked_slope[p_vals > 0.05] = 0

        if (masked_slope != 0).sum() == 0:
            print(f'  No significant voxels: {task} / {contrast}')
            continue

        vmax = np.percentile(np.abs(masked_slope[masked_slope != 0]), 95)
        slope_img = masker.inverse_transform(masked_slope)
        display = plotting.plot_stat_map(
            slope_img, display_mode='z', cut_coords=6,
            cmap='RdBu_r', vmax=vmax, colorbar=True,
            title=f'Group slope (p<0.05): {task} | {contrast}',
        )
        safe = contrast.replace('/', '-')
        display.savefig(group_fig_dir / f'{task}_{safe}.png')
        plt.close('all')

print(f'Group slope maps saved to {group_fig_dir}')